# バンドリング研究　プログラムパイプライン

論文で用いる計算を実行するプログラム群を上から順に実行しながら確認するためのノートブック。

計算そのものはすべて `scripts/` 配下のプログラムが行い、このノートブックは
それらを順番に呼び出して、入力・出力・途中の値がつながっているかを確かめる。
ノートブック側に計算式は書かれていないので、ここを読めば
「どのプログラムが何を受け取り、何を出し、次のどこへ渡るか」がわかる。

## 使い方

- **1セルずつ上から順に実行する**（「すべて実行」でも同じ結果になる）。
  セルは上から下へ一方向にしか依存しないので、戻って実行し直す必要はない。
- 各確認は `check(...)` が ✅ / ❌ を出し、最後の章で一覧表にまとめる。
- 出力先は既定では作業用フォルダ `outputs/walkthrough/<実行時刻>/` で、
  論文が参照するファイルには触れない。論文用のファイルを更新したいときは
  下の `UPDATE_CANONICAL` と `SYNC_TEX_FIGURES` を使う。

## 動作を切り替えるスイッチ

いずれもノートブック内で定義しており、値を書き換えて実行し直せば切り替わる。

| スイッチ | 既定 | True にすると |
|---|---|---|
| `RUN_FROM_RAW` | `False` | 点検データの取得からマルコフ推定用データの作成までを実行する（第2章） |
| `RUN_GUROBI` | `False` | 地域分割最適化を解き直す（Gurobiが必要。第5章） |
| `RUN_MAPS` | `True` | 地図を描く（geopandasが必要。第6章） |
| `UPDATE_CANONICAL` | `False` | 図表を作業用フォルダではなく `figures/`・`outputs/` に出力する |
| `SYNC_TEX_FIGURES` | `False` | 図を `main.tex` が読む場所へコピーする（第10章） |

## 0. セットアップ

以降のすべてのセルが使う土台を用意する。ここで行うのは次の4つ。

1. **リポジトリルートの特定** — `notebooks/` から開いてもリポジトリ直下から開いても
   同じように動くよう、`scripts/` フォルダの有無を手がかりに位置を割り出す。
   分析コード本体（`src/`）を読み込めるようにパスも通す。
2. **出力先の決定** — 実行のたびに `outputs/walkthrough/<実行時刻>/` を作る。
   実行ごとに別フォルダなので、過去の結果が上書きされない。
3. **図表の出力先の選択** — `UPDATE_CANONICAL` で、作業用フォルダに出すか、
   論文が参照する `figures/`・`outputs/` を直接更新するかを切り替える。
4. **共通の道具の定義** — 以降のセルはこの3つだけを使う。

| 関数 | 役割 |
|---|---|
| `run(スクリプト名, 引数...)` | `scripts/` のプログラムを1本実行し、出力をそのまま表示する。失敗したら止まる |
| `check(段階, 内容, 判定, 補足)` | 確認結果を記録して ✅ / ❌ を表示する。最後の一覧表の材料になる |
| `show(画像パス)` | 生成された図をノートブック内に表示する |

In [ ]:
from __future__ import annotations

import datetime
import json
import os
import pickle
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

# --- 1. リポジトリの位置を割り出す ------------------------------------------
# notebooks/ から開いてもリポジトリ直下から開いても同じように動くよう、
# scripts/ フォルダが見えるところまで1階層さかのぼる。
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scripts").is_dir():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "scripts").is_dir(), f"リポジトリルートが見つからない: {Path.cwd()}"
# 分析コード本体（src/bundling_analysis）を import できるようにする
sys.path.insert(0, str(REPO_ROOT / "src"))

# --- 2. この実行専用の出力フォルダを作る ------------------------------------
# 実行時刻でフォルダを分けるので、前回の結果を上書きしない。
# outputs/ は .gitignore 済みなので、何度実行してもリポジトリは汚れない。
PY = sys.executable
RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT = REPO_ROOT / "outputs" / "walkthrough" / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)

# --- 3. 図表の出力先を決める ------------------------------------------------
# 既定では上の作業用フォルダに出力し、論文が参照するファイルには触れない。
# 論文用の図表を作り直すときだけ UPDATE_CANONICAL = True にする。
UPDATE_CANONICAL = False
FIG_DIR = (REPO_ROOT / "figures") if UPDATE_CANONICAL else OUT
TAB_DIR = (REPO_ROOT / "outputs") if UPDATE_CANONICAL else OUT
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

# --- 4. 以降のセルが使う共通の道具 ------------------------------------------
CHECKS: list[dict] = []   # check() の記録がたまり、最後の章で一覧表になる


def check(stage: str, name: str, passed: bool, detail: str = "") -> bool:
    """確認結果を1件記録して ✅ / ❌ を表示する。"""
    CHECKS.append({"stage": stage, "check": name, "result": "PASS" if passed else "FAIL",
                   "detail": detail})
    print(f"{'✅' if passed else '❌'} [{stage}] {name}" + (f" — {detail}" if detail else ""))
    return passed


def run(script: str, *args, timeout: int = 1800) -> str:
    """scripts/ 配下のプログラムを1本実行し、その出力をそのまま表示する。

    ターミナルで実行するのと同じ形で呼び出すので、ノートブック側の都合で
    計算結果が変わることはない。失敗したらその場で例外を出して止める。
    """
    cmd = [PY, str(REPO_ROOT / "scripts" / script), *map(str, args)]
    env = os.environ.copy()
    # 子プロセスからも src/ を import できるようにする
    src = str(REPO_ROOT / "src")
    env["PYTHONPATH"] = src + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
    print(f"$ python scripts/{script} " + " ".join(map(str, args)))
    r = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True, env=env, timeout=timeout)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr, file=sys.stderr)
        raise RuntimeError(f"{script} が終了コード {r.returncode} で失敗")
    return r.stdout


def show(path, width: int = 760) -> None:
    """生成された図をノートブック内に表示する。"""
    p = Path(path)
    display(Image(filename=str(p), width=width)) if p.exists() else print(f"(未生成: {p})")


print("REPO_ROOT :", REPO_ROOT)
print("python    :", PY)
print("作業用出力:", OUT.relative_to(REPO_ROOT))
print("図の出力先:", FIG_DIR.relative_to(REPO_ROOT), "/ 表の出力先:", TAB_DIR.relative_to(REPO_ROOT))
print("UPDATE_CANONICAL =", UPDATE_CANONICAL,
      "（True なら figures/・outputs/ の正本を更新する）")

### 0-2. 実行環境の確認

必要なライブラリが揃っているかを確認する。用途は次のとおり。

- `numpy` / `pandas` — 表形式データの読み書きと集計
- `scipy` — マルコフ推定の数値計算
- `matplotlib` — 作図
- `geopandas` / `shapely` — 行政界データの処理と地図描画
- `gurobipy` — 地域分割最適化を解くソルバー

このうち **`gurobipy` だけは無くても構わない**。有無に応じて第5章の動作が変わる。

- **ある場合** — `RUN_GUROBI = True` にすれば、地域分割最適化を実際に解き直せる。
- **ない場合** — 最適化は解かず、保存済みの結果ファイルどうしの整合性だけを確認する。
  この状態でもノートブックは最後まで完走する。

判定結果は `has_gurobi` に入り、第5章がこれを見て分岐する。

In [ ]:
# 必要なライブラリが import できるかを1つずつ確かめる
import importlib

rows = []
for mod in ["numpy", "pandas", "scipy", "matplotlib", "geopandas", "shapely", "gurobipy"]:
    try:
        m = importlib.import_module(mod)
        rows.append({"module": mod, "status": "OK", "version": getattr(m, "__version__", "")})
    except Exception as e:
        rows.append({"module": mod, "status": f"NG ({type(e).__name__})", "version": ""})

env = pd.DataFrame(rows)
display(env)

# gurobipy 以外が揃っていれば、推定・集計・作図はすべて実行できる
need = env[env.module.isin(["numpy", "pandas", "scipy", "matplotlib", "geopandas"])]
check("0.環境", "図表生成に必要なライブラリが揃っている",
      bool((need.status == "OK").all()), f"python {sys.version.split()[0]}")

# gurobipy の有無で第5章の動作が変わる。無くても最後まで実行できる。
has_gurobi = env.set_index("module").loc["gurobipy", "status"] == "OK"
print("\nGurobi:", "あり → 第5章で地域分割最適化を解き直せる" if has_gurobi
      else "なし → 第5章は保存済み結果の整合性確認のみ（このノートブックは完走する）")

### 0-3. 入力・出力・パラメータの指定方法

各プログラムは、入力ファイル・出力先・計算条件を**コマンドライン引数**で受け取る。
プログラム本体を書き換えなくても、引数を変えるだけで対象データや条件を差し替えられる。
このノートブックも同じ仕組みで呼び出しているだけなので、
ここで渡している引数を変えれば、そのまま別の条件で計算できる。

書き方は次のとおり。

```python
run("プログラム名.py",
    "--入力の指定", 入力ファイルのパス,
    "--条件の指定", 値,
    "--出力先の指定", 出力ファイルのパス)
```

引数を省略した項目には、各プログラムに定められた既定値が使われる。
既定値は「論文の結果をそのまま再現する組み合わせ」になっているので、
何も指定しなければ論文と同じ計算になる。

どんな引数を受け付けるかは、`--help` を渡すと一覧で確認できる。
プログラムの中身を読まなくても、指定できる項目・既定値・意味がわかる。

下のセルでは、実際に次の2つを行う。

1. 期待契約件数の感度分析プログラムに `--help` を渡し、指定できる引数を一覧表示する
2. 同じプログラムに既定と異なる条件を与えて、別の図を作る
   （橋梁数の上限を100に、同時発注上限を2と5に変更）

2の出力は作業用フォルダに書かれるため、論文用の図は変化しない。

In [ ]:
# (1) 指定できる引数の一覧を見る。プログラムの中身を読まずに使い方がわかる。
run("plot_expected_contracts_scaling_analysis.py", "--help")

# (2) 既定と異なる条件を与えて図を作る。
#     --max-n         横軸（地域内の橋梁数）の上限。既定322 → 100
#     --bundle-limits 同時発注上限Lの候補。既定[1,3,5,7,10] → [2,5]
#     --output-stem   出力先。作業用フォルダを指定するので論文用の図は変わらない
run("plot_expected_contracts_by_limit.py",
    "--max-n", 100,
    "--bundle-limits", 2, 5,
    "--output-stem", OUT / "sandbox_by_limit")
show(OUT / "sandbox_by_limit.png", width=520)
print("→ プログラムを書き換えずに、引数だけで条件を変えられる。")

## 1. パイプライン全体像

点検データから論文の図表までのつながりは次のとおり。

```
[x-Road原データ] ─step3_extract_rc_bridges─▶ 宮城県RC橋
       │                                        │
       │                        step3_filter_target_municipalities
       │                                        ▼
       │                        data/processed/target_rc_bridges_322.csv ──┐
       │                                        │                          │
[道路メンテナンス年報]                          │                          │
       └──step3_prepare_markov_input────▶ markov_input_*.txt               │
                                                │                          │
                       ┌────────────────────────┼──────────────┐           │
                       ▼                        ▼              ▼           │
            make_transition_counts   step3_run_emarkov  plot_inspection_interval
              (健全度遷移の集計表)          │              (点検間隔の分布図)
                                           ▼
                              推移確率行列 P → q = 0.0123298
                                           │
              data/processed/emarkov_20251207_200558/（この出力を保存し、下流が読む）
                                           │
                    ┌──────────────────────┴───────────────────┐
                    ▼                                          ▼
        expected_contracts(N,L,q)                    step3_build_distance_matrix
        （期待契約件数の唯一の実装）                  → distance_matrix_322.pkl
                    │                                          │
                    │                                          ▼
                    │                              run_gurobi_districting（Gurobi必須）
                    │                                → 最適化結果CSV + 割当pkl
                    │                                          │
                    │                        reevaluate_optimization_objectives
                    │                                          ▼
                    │                     optimization_results_exact_objective.csv
                    │                                          │
   ┌────────────────┼──────────────────┬───────────────────────┼──────────────┐
   ▼                ▼                  ▼                       ▼              ▼
plot_expected_   plot_expected_   plot_optimization_    plot_dm_sensitivity  make_
contracts_       contracts_       results               plot_region_          districting_
by_limit         scaling_analysis (期待契約件数の図表)   breakdown            maps
                                                                             (地図・atlas)
```

### `RUN_FROM_RAW = False` のとき固定される値

上の図の上半分（点検データ → 対象橋梁 → マルコフ推定用データ）を省略すると、
そこで決まる次の値が**保存済みのまま固定**される。

| 値 | 意味 | 変えるには |
|---|---|---|
| $q$ = 0.012329787974114258 | 1橋が1年間に補修需要を出す確率 | 点検データかマルコフ推定用データを変えて再推定する |
| $N$ = 322 | 対象橋梁数 | 対象地域や抽出条件を変えて再抽出する |
| 橋梁間距離 | 地域内最大距離の制約に使う | 対象橋梁が変われば作り直す |
| $L$ = 5 | 1件の契約にまとめられる上限橋梁数 | 最適化を解き直す |

つまり、**条件を変えずに再実行するだけなら、これらの値は変わらない**。
逆に条件を変えたいときは、値を書き換えるのではなく
`RUN_FROM_RAW = True` にして上流から計算し直す必要がある。
$q$ については、再推定した結果が保存され、それを下流が読む仕組みになっている（第3章）。

### 1-2. どこから計算し直すかを決める（`RUN_FROM_RAW`）

上の図の**上半分を実行するかどうか**を決める。上半分とは、
点検データを読み込んで対象橋梁を絞り込み、マルコフ推定用データを作るまでの工程である。

| | `RUN_FROM_RAW = False`（既定） | `RUN_FROM_RAW = True` |
|---|---|---|
| 開始点 | 保存済みの対象橋梁CSVとマルコフ推定用データ | 点検データそのもの |
| 実行する工程 | 図の下半分（推定・最適化・作図） | 全工程 |
| 所要 | 約2分 | 約4分 |
| 使いどころ | 論文の結果を確認する。作図や集計を変える | 対象地域を変える。点検データを更新する。抽出条件を見直す |

工程の上半分は結果が変わらない限り繰り返す必要がないため、
その出力（対象橋梁CSV・マルコフ推定用データ）をリポジトリに保存してある。
既定ではそれを使うので、点検データの原本が手元になくても下半分は動く。

#### `True` にする場合に必要なデータ

点検データの原本はサイズが大きくリポジトリには含めていないので、
別に置いた場所を `EXTERNAL_ROOT` で指定する。必要なのは次の3つ。

| データ | 役割 | 既定のファイル名 |
|---|---|---|
| x-Road原データ | 全国道路施設点検データベースの抽出結果。橋梁の諸元・位置・健全度 | `data/input/prefecture_04_宮城県_12425records_original.csv` |
| 行政界shp | 国土数値情報の行政区域データ。座標から市区町村を判定する | `data/N03-20230101_GML/N03-23_230101.shp` |
| 道路メンテナンス年報（施設番号付与済み） | 点検履歴。橋梁ごとの健全度の推移 | `data/maintenance_prefecture_exports_with_shisetsu/04_宮城県.csv` |

`EXTERNAL_ROOT` は次の順に探し、最初に見つかったものを使う。

1. 環境変数 `BUNDLING_EXTERNAL_ROOT`
2. リポジトリ内の `data/external/`
3. このマシンでの既知の置き場所

3つのうち1つでも欠けていれば `RUN_FROM_RAW` は自動的に `False` に戻るので、
データが無い環境で `True` にしてもエラーにはならず、そのまま最後まで実行できる。

In [ ]:
# ここを True にすると原データから回す（データが無ければ自動で False に落ちる）
RUN_FROM_RAW = False

# 外部データの置き場所。パスを直書きせず、候補の中から見つかったものを使う。
_candidates = [
    Path(os.environ["BUNDLING_EXTERNAL_ROOT"]) if os.environ.get("BUNDLING_EXTERNAL_ROOT") else None,
    REPO_ROOT / "data/external",
    # このマシンでの既知の置き場所（notes/pre_git_migration_inventory.md に記録あり）
    Path.home() / "Library/CloudStorage/OneDrive-個人用/02_東北大学/03_博士課程"
                  "/2025FY/03_研究テーマ検討/20251208_定期打ち合わせ",
]
EXTERNAL_ROOT = next((p for p in _candidates if p is not None and p.is_dir()), None)

EXTERNAL = {}
if EXTERNAL_ROOT is not None:
    EXTERNAL = {
        "x-Road原データ": EXTERNAL_ROOT / "data/input/prefecture_04_宮城県_12425records_original.csv",
        "行政界shp": EXTERNAL_ROOT / "data/N03-20230101_GML/N03-23_230101.shp",
        "年報CSV(施設番号付与済)": EXTERNAL_ROOT / "data/maintenance_prefecture_exports_with_shisetsu/04_宮城県.csv",
    }

print("外部データのルート:", EXTERNAL_ROOT or "見つからず")
if EXTERNAL:
    display(pd.DataFrame([
        {"データ": k, "有無": "OK" if v.exists() else "無し",
         "サイズ": f"{v.stat().st_size:,}" if v.exists() else "-"}
        for k, v in EXTERNAL.items()
    ]))

missing = [k for k, v in EXTERNAL.items() if not v.exists()] if EXTERNAL else ["(ルート未検出)"]
if RUN_FROM_RAW and missing:
    RUN_FROM_RAW = False
    print(f"原データが揃わないため RUN_FROM_RAW = False に切替: {missing}")
print(f"\nRUN_FROM_RAW = {RUN_FROM_RAW}"
      + ("（原データから回す）" if RUN_FROM_RAW else "（中間生成物から開始）"))

## 2. ステージ1 — 点検データから対象橋梁を絞り込む

論文が分析対象とするのは、**宮城県南部の6市町（七ヶ宿町・白石市・蔵王町・川崎町・村田町・大河原町）が
管理するRC橋322橋**である。地域分割最適化はこの322橋をいくつの管理エリアに分けるかを決める問題として解く。

これとは別に、劣化の推移確率は**宮城県内のRC橋5,525橋**から推定する。
6市町だけでは健全度が変化した観測数が足りないため、
同じ点検制度の下にある県内のRC橋全体から共通の推移確率を求め、それを322橋へ適用している。

絞り込みは2段階で行う。

1. `step3_extract_rc_bridges.py` — 点検データから宮城県内のRC橋を抽出する（5,525橋）
2. `step3_filter_target_municipalities.py` — 対象6市町が管理する橋に絞る（322橋）

### 2-1. 保存済みの対象橋梁CSVを確認する

`RUN_FROM_RAW = False` のときは上の2工程を実行せず、保存済みの
`data/processed/target_rc_bridges_322.csv` をそのまま使う。
ただし黙って使うのではなく、**下流の計算が前提としている条件を満たしているか**をここで確かめる。

| 確認すること | これが崩れると何が起きるか |
|---|---|
| 322行ちょうど | 対象橋梁数が変わる。期待契約件数も最適化の答えも変わる |
| 施設番号に重複が無い | 同じ橋を二重に数え、補修需要を過大に見積もる |
| 対象6市町のみが含まれる | 分析対象がずれる |
| 管理者別の橋梁数が既知の内訳と一致 | 現行管理を基準にした比較値が変わる |
| 緯度経度が対象地域の範囲に収まる | 橋梁間距離が狂い、距離制約が意味を失う |

これらがすべて ✅ なら、保存済みCSVは論文執筆時と同じものだと確認できたことになる。
なお322という数や6市町の名前は現在の対象地域に固有の値なので、
対象地域を変えたときはこの確認も作り直す必要がある。

In [ ]:
# 保存済みの対象橋梁CSVを読み、下流の計算が前提としている条件を満たすか確かめる
bridges_csv = REPO_ROOT / "data/processed/target_rc_bridges_322.csv"
bridges = pd.read_csv(bridges_csv)

print("列:", list(bridges.columns))
counts = bridges["管理者"].value_counts()   # 管理者＝市町。現行管理の基準値の計算に使う
display(counts.to_frame("橋梁数"))

# 対象橋梁数。期待契約件数も最適化の答えもこの数に依存する
check("1.対象橋梁", "322行ちょうど", len(bridges) == 322, f"{len(bridges)} 行")

# 同じ橋を二重に数えていないこと
check("1.対象橋梁", "施設番号に重複が無い",
      bridges["shisetsu_bangou"].is_unique, f"unique={bridges['shisetsu_bangou'].nunique()}")

# 分析対象が対象6市町からずれていないこと
check("1.対象橋梁", "対象6市町のみ", set(counts.index) == {"大河原町", "白石市", "蔵王町", "村田町", "川崎町", "七ヶ宿町"},
      str(counts.to_dict()))

# 管理者別の内訳。現行管理を基準にした比較値（2.5883件）はこの内訳から計算される
check("1.対象橋梁", "管理者別内訳が data/README.md の記載と一致",
      counts.to_dict() == {"大河原町": 103, "白石市": 98, "蔵王町": 64, "村田町": 33, "川崎町": 14, "七ヶ宿町": 10})

# 座標。橋梁間距離を経由して距離制約に効くため、範囲外の値が混ざっていないか見る
check("1.対象橋梁", "緯度経度が宮城県南部の範囲に収まる",
      bool(bridges["緯度"].between(37.7, 38.4).all() and bridges["経度"].between(140.2, 141.0).all()),
      f"lat {bridges['緯度'].min():.3f}–{bridges['緯度'].max():.3f}, "
      f"lon {bridges['経度'].min():.3f}–{bridges['経度'].max():.3f}")

### 2-2. 点検データから対象橋梁を作り直す　🟢`RUN_FROM_RAW = True` のときだけ

点検データを読み込み、対象橋梁CSVを新しく作る工程。
`RUN_FROM_RAW = False` のときは何もしない。

**ここで作られるCSVが、以降の分析の出発点になる新しいデータである。**
対象地域を変える場合（例: 別の県や、市町を追加する場合）は、
この工程を実行して得られたCSVを保存済みCSVと差し替えることになる。

処理は2段階。

1. **RC橋の抽出** — 点検データから構造形式がRCの橋を選び、緯度経度・健全度・管理者を整理する
2. **行政界によるクリップ** — 抽出した橋の座標を行政区域データと照合し、対象都道府県の外に落ちる橋を除く

行政界クリップを行う理由は、点検データに記録された座標が必ずしも管理者の所在と
一致しないためである。実際、白石市が管理する橋のうち1件は座標が福島県側にあり、
この照合で除外される。これを省くと対象橋梁数が変わってしまう。

#### 対象を広げるときに見直す条件

RC橋に限るのは本研究の設定であり、固定された前提ではない。
判定は構造区分コードの百の位が3かどうかで行っており、
`step3_extract_rc_bridges.py` の `filter_rc_bridges()` に書かれている。
コマンドライン引数では変えられないので、対象構造を広げるときはこの関数を修正する。
対象都道府県は `--prefecture` で指定できる。

#### 保存済みCSVとの比較

最後に、できたCSVと保存済みCSVを比べる。

- **同じ橋の集合になった場合** — 点検データから同じ結果が再現できたことになる
- **異なる集合になった場合** — 対象地域や抽出条件を変えたのなら、それが期待どおりの動作である

この比較は「どちらが正しいか」の判定ではなく、**何が変わったかを知るための情報**として表示する。
判定として確認するのは、下流の計算が動くためにどの地域でも必要な条件だけに限る。

In [ ]:
RAW_OUT = OUT / "from_raw"

if RUN_FROM_RAW:
    RAW_OUT.mkdir(parents=True, exist_ok=True)

    # 1. 点検データからRC橋を抽出し、行政界で対象都道府県の外を除く
    run("step3_extract_rc_bridges.py",
        "--input", EXTERNAL["x-Road原データ"],
        "--output", RAW_OUT / "miyagi_rc_bridges.csv",
        "--boundary", EXTERNAL["行政界shp"])

    # 2. 対象市町が管理する橋だけに絞る
    run("step3_filter_target_municipalities.py",
        "--input", RAW_OUT / "miyagi_rc_bridges.csv",
        "--output", RAW_OUT / "target_rc_bridges_322.csv")

    regen = pd.read_csv(RAW_OUT / "target_rc_bridges_322.csv")
    print(f"作り直した対象橋梁: {len(regen)} 件")
    display(regen["管理者"].value_counts().to_frame("橋梁数"))

    # --- 判定: 対象地域が変わっても成り立つべき条件だけを見る ---
    check("1.対象橋梁", "作り直したCSVに橋梁が1件以上ある", len(regen) > 0, f"{len(regen)} 件")
    check("1.対象橋梁", "施設番号に重複が無い", regen["shisetsu_bangou"].is_unique)
    check("1.対象橋梁", "緯度経度に欠損が無い",
          bool(regen[["緯度", "経度"]].notna().all().all()))

    # --- 情報: 保存済みCSVとの違い。地域を変えたなら違って当然なので判定にはしない ---
    new_ids, old_ids = set(regen["shisetsu_bangou"]), set(bridges["shisetsu_bangou"])
    if new_ids == old_ids:
        print("\n→ 保存済みCSVと同じ橋の集合。点検データから同じ結果が再現できた。")
    else:
        print(f"\n→ 保存済みCSVとは異なる集合（新規 {len(new_ids - old_ids)} 件 / "
              f"今回含まれなかった {len(old_ids - new_ids)} 件）。")
        print("  対象地域や抽出条件を変えたのであれば、これが期待どおりの結果。")
        print("  これを以降の分析の入力にするには、保存済みCSVと差し替える:")
        print(f"    cp {(RAW_OUT / 'target_rc_bridges_322.csv')}"
              f" data/processed/target_rc_bridges_322.csv")
        print("  ※ 差し替えたら橋梁間距離の作り直しと最適化の解き直しも必要になる。")
else:
    print("RUN_FROM_RAW = False のためスキップ（保存済みの対象橋梁CSVを使う）")

## 3. ステージ2 — 劣化の推定から補修需要発生確率 $q$ へ

論文の数値はすべて、1橋が1年間に補修需要を出す確率 $q$ に依存する。
この章はその $q$ がどこから来ているかを扱う。工程は3層に分かれる。

| 層 | 処理 | 入力 → 出力 | このノートでの扱い |
|---|---|---|---|
| ① 推定用データの作成 | `step3_prepare_markov_input.py` | 対象橋梁CSV＋道路メンテナンス年報 → `markov_input_*.txt` | `RUN_FROM_RAW = True` のとき実行 |
| ② 推定 | `step3_run_emarkov.py` | `markov_input_*.txt` → 推移確率行列・$q$ | **毎回その場で再推定する**（約1.5秒） |
| ③ 利用 | `expected_contracts.py` | 推移確率行列 → $q$ → 全図表 | **②が保存した結果を読む** |

### 変更に気づくための仕組み

③が読むファイルを差し替えれば $q$ は変わり、図表も最適化結果もすべて変わる。
それが気づかないうちに起きないよう、`expected_contracts.py` には
**論文が依拠した行列がピン留め値として併記**してある。
読み込んだファイルがピン留め値とずれていれば、読み込んだ時点で警告が出る。

更新は「②を実行し直す → 出力を保存する → ピン留め値を更新する」を揃えて初めて完了し、
そのとき `tests/test_transition_matrix_provenance.py` が通るようになる。

推定計算に乱数は使われていないため、同じ推定用データからは必ず同じ行列が得られる。
保存された行列は、保存された推定用データから一意に決まる値である。

なお本研究で採用しているのは `with_supply_collapse`（健全度IVをIIIへ統合した系列）である。

In [ ]:
CANON_MARKOV = REPO_ROOT / "data/processed/markov_input_20251207_200558"
SCENARIOS = ["without_supply", "without_supply_collapse", "with_supply", "with_supply_collapse"]

if RUN_FROM_RAW:
    # 対象橋梁CSVと道路メンテナンス年報を突合し、4シナリオの推定用データを作る
    run("step3_prepare_markov_input.py",
        "--rc-bridges", RAW_OUT / "miyagi_rc_bridges.csv",
        "--maintenance", EXTERNAL["年報CSV(施設番号付与済)"],
        "--output-dir", RAW_OUT / "markov_input")
    MARKOV_DIR = RAW_OUT / "markov_input"

    def _body(path):
        # 改行コードの違い（Windows生成はCRLF、Mac生成はLF）を無視して中身を比べる
        return path.read_bytes().replace(b"\r\n", b"\n")

    same = {s: _body(MARKOV_DIR / f"markov_input_{s}.txt") == _body(CANON_MARKOV / f"markov_input_{s}.txt")
            for s in SCENARIOS}
    display(pd.Series(same, name="保存済みデータと内容一致").to_frame())
    TSV_MATCHES_CANONICAL = all(same.values())
    check("2.劣化推定", "作り直した推定用データ4本が保存済みのものと内容一致",
          TSV_MATCHES_CANONICAL, f"{sum(same.values())}/{len(same)} ファイル")
else:
    MARKOV_DIR = CANON_MARKOV
    TSV_MATCHES_CANONICAL = True   # 保存済みデータそのものを使うので定義上一致
    print("RUN_FROM_RAW = False のためスキップ（保存済みの推定用データを使う）")

print("\nこの後の推定に使う入力:", MARKOV_DIR.relative_to(REPO_ROOT))
for p in sorted(MARKOV_DIR.glob("*.txt")):
    print(f"  {p.name}  ({p.stat().st_size:,} bytes)")

### 3-2. 推定を実行する

前のセルで決めた入力（`MARKOV_DIR`）から推移確率行列と $q$ を推定する。

出力先はこの実行の作業フォルダ `OUT/emarkov/` で、**論文が使う値はこのセルでは変わらない**。
推定を実行しただけで論文の数値が置き換わると、確認のつもりの実行で結果が動いてしまうためである。
まず推定し、次のセルで既存の値と突き合わせ、**違いを確認したうえで採用する**という順序にしてある。

対象や条件を変えて $q$ を更新したい場合は、次のセルが表示する採用手順に従い、
出力先を `data/processed/emarkov_20251207_200558/` にして実行し直す。
そのファイルを③が読むので、以降の計算すべてに反映される。

In [ ]:
emarkov_out = OUT / "emarkov"
run("step3_run_emarkov.py",
    "--input-dir", MARKOV_DIR,
    "--scenarios", "with_supply_collapse",
    "--output-dir", emarkov_out)

### 3-3. 3つの値を突き合わせる

ここで比べるのは次の3つ。すべて推移確率行列（から導かれる $q$）である。

| 呼び方 | 実体 | 意味 |
|---|---|---|
| 再推定 | 前のセルで推定用データから計算した値 | いま手元のデータから得られる値 |
| 保存済み | `data/processed/emarkov_.../..._stage3.csv` | ③が読み込み、下流の全計算に使われる値 |
| ピン留め | `expected_contracts.py` の `OPTIMIZATION_TRANSITION_MATRIX` | 論文が依拠している値 |

3つが一致していれば、「点検データ → 推定 → 図表」の鎖が途切れていないことになる。
一致しない場合は、どこがずれたかによって意味が変わる。

| 状況 | 意味 | 対応 |
|---|---|---|
| 保存済み ≠ ピン留め | 保存された行列が、論文が依拠する値から変わっている | 意図した更新ならピン留め値も更新する。違うならファイルを戻す |
| 再推定 ≠ 保存済み（推定用データは同じ） | 同じデータから違う答えが出た＝推定プログラムが変わった | 推定側の変更を確認する |
| 再推定 ≠ 保存済み（推定用データも違う） | 入力が変わったので $q$ も変わる。正常な更新の途中 | 保存済みファイルを更新し、ピン留め値も更新する |
| 再推定 = 保存済み だが 推定用データは違う | 入力が変わったのに $q$ が変わっていない | 反映漏れの可能性。保存先のパスを確認する |

In [ ]:
# 再推定・保存済み・ピン留めの3つを突き合わせる
from bundling_analysis.expected_contracts import (
    CANONICAL_TRANSITION_MATRIX_PATH,
    DEFAULT_TRANSITION_MATRIX,
    OPTIMIZATION_TRANSITION_MATRIX,
    expected_contracts,
    load_transition_matrix,
    repair_probability_from_transition_matrix,
)

scen = emarkov_out / "with_supply_collapse"
rerun_P = np.loadtxt(scen / "with_supply_collapse_transition_matrix_stage3.csv", delimiter=",")
rerun_q = float(json.loads((scen / "with_supply_collapse_repair_probability.json").read_text())["q"])

file_P = np.array(load_transition_matrix(CANONICAL_TRANSITION_MATRIX_PATH))  # ③が読む値
pin_P = np.array(OPTIMIZATION_TRANSITION_MATRIX)                             # 論文が依拠する値
used_P = np.array(DEFAULT_TRANSITION_MATRIX)                                 # 実際に下流が使う値

_, file_q = repair_probability_from_transition_matrix(load_transition_matrix(CANONICAL_TRANSITION_MATRIX_PATH))
_, pin_q = repair_probability_from_transition_matrix(OPTIMIZATION_TRANSITION_MATRIX)
_, used_q = repair_probability_from_transition_matrix(DEFAULT_TRANSITION_MATRIX)

display(pd.DataFrame({
    "実体": {"再推定": "いま推定用データから計算した値",
             "保存済み": str(CANONICAL_TRANSITION_MATRIX_PATH.relative_to(REPO_ROOT)),
             "ピン留め": "expected_contracts.OPTIMIZATION_TRANSITION_MATRIX",
             "下流が使う値": "expected_contracts.DEFAULT_TRANSITION_MATRIX"},
    "q（全桁）": {"再推定": repr(rerun_q), "保存済み": repr(file_q),
                 "ピン留め": repr(pin_q), "下流が使う値": repr(used_q)},
}))

# ③がファイルを読んでいること（数値を直接書き込んでいないこと）
check("2.劣化推定", "下流が使う行列が、保存済みファイルの内容と一致",
      bool((used_P == file_P).all()), "expected_contracts はファイルを読んでいる")

# 保存済みファイルが論文の値と一致していること
check("2.劣化推定", "保存済みファイルが論文のピン留め値と一致",
      bool((file_P == pin_P).all()),
      f"最大差 {np.abs(file_P - pin_P).max():.2e}")

# ② → ③ の接続そのもの。推定用データから再推定した結果が保存済みファイルと一致するか
d_rerun = float(np.abs(rerun_P - file_P).max())
check("2.劣化推定", "推定用データから再推定した行列が保存済みファイルと一致",
      d_rerun == 0.0, f"最大差 {d_rerun:.2e}")

# 状況の解釈（前のセルの TSV_MATCHES_CANONICAL と組み合わせる）
if d_rerun != 0.0 and TSV_MATCHES_CANONICAL:
    print("\n⚠ 推定用データは保存済みと同じなのに、推定結果が違う。"
          "推定プログラムが変わった可能性がある。")
elif d_rerun != 0.0:
    print("\n⚠ 入力が変わったため q も変わった。保存済みファイルとピン留め値の更新が必要:")
    print(f"    python scripts/step3_run_emarkov.py --input-dir {MARKOV_DIR}"
          f" --scenarios with_supply_collapse"
          f" --output-dir {CANONICAL_TRANSITION_MATRIX_PATH.parents[1]}")
elif not TSV_MATCHES_CANONICAL:
    print("\n⚠ 推定用データが変わったのに q が変わっていない。反映漏れが無いか確認すること。")

q = used_q   # 以降のセルはこの q を使う（＝下流の全計算と同じ値）
q_rerun = rerun_q

### 3-4. 同じ入力から作る表と図

推定用データは、推定だけでなく論文の表と図の材料にもなっている。
ただし**読むファイルが別**なので、そこを取り違えていないかをここで確かめる。

| 出力 | プログラム | 読むファイル |
|---|---|---|
| 健全度遷移の集計表 | `make_transition_counts.py` | `markov_input_with_supply.txt`（健全度IVを残したまま） |
| 点検間隔の分布図 | `plot_inspection_interval.py` | `markov_input_with_supply_collapse.txt`（IVをIIIへ統合） |
| 推移確率行列・$q$ | `step3_run_emarkov.py` | `markov_input_with_supply_collapse.txt` |

表はIV統合前のデータから作る。論文ではIVを統合する前の遷移状況を示したうえで、
推定ではIIIへ統合したと説明するためである。

そこで、**表の材料と推定の材料で遷移件数が一致するか**を確認する。
一致していれば、別々のファイルを読みながらも同じ時点のデータを見ていることになる。
比較する件数は前のセルの推定記録から読み取るので、数値を直接書き込んではいない。

ここもう解決しているならいらなくない？

In [ ]:
# 表と図を作る。どちらも MARKOV_DIR の中の別々のシナリオファイルを読む。
run("make_transition_counts.py",
    "--input", MARKOV_DIR / "markov_input_with_supply.txt",
    "--output", TAB_DIR / "transition_counts.csv")
run("plot_inspection_interval.py",
    "--input", MARKOV_DIR / "markov_input_with_supply_collapse.txt",
    "--output-stem", FIG_DIR / "inspection_interval")

tc = pd.read_csv(TAB_DIR / "transition_counts.csv", index_col=0)
display(tc)

# 推定に使った遷移件数は、前のセルの実行記録に残っている
run_meta = json.loads((emarkov_out / "emarkov_run.meta.json").read_text())
n_estimated = int(run_meta["results"][0]["n_transitions"])
n_table = int(tc.to_numpy().sum())

check("2.劣化推定", "表の材料と推定の材料で遷移件数が一致",
      n_table == n_estimated, f"表 {n_table:,} 件 / 推定 {n_estimated:,} 件")
show(FIG_DIR / "inspection_interval.png", width=620)

## 4. ステージ3 — 橋梁間の距離行列

地域分割最適化では「同じ管理エリアに入る橋どうしの距離が上限 $D$ 以内」という制約を課す。
そのために、対象橋梁の総当たりの距離行列を用意する。

距離は緯度経度から求める**大円距離**（haversine、地球半径6371.0088km）である。
道路経路長ではないため、外部サービスへの接続は必要なく、手元だけで計算できる。

計算結果はSQLiteのキャッシュに蓄積され、同じ座標の組は再計算せずに済むようになっている。
キャッシュの場所は次の順に決まる（`src/bundling_analysis/distance_cache.py`）。

1. 環境変数 `DIST_CACHE_DB_PATH`
2. `03_研究テーマ検討/dist_cache/dist_cache.sqlite`（研究フォルダ構成が検出できる場合）
3. `results/dist_cache.sqlite`（旧配置）

共有キャッシュは他の分析と共用で数GB規模になるため、
このノートブックでは書き込みを避け、実行ごとの一時キャッシュを使う。

### 4-1. 保存済みの距離行列を確認する

`RUN_FROM_RAW = False` のときは、保存済みの
`data/processed/distance_matrix_322_20251208.pkl` をそのまま使う。
中身は次の2つで、下流は `order` を手がかりに行列と橋梁を対応づける。

| キー | 内容 |
|---|---|
| `order` | 行・列の並び順を表す橋梁IDの一覧 |
| `d_core` | 距離行列本体（km） |

確認するのは、**距離行列として成立しているか**と**対象橋梁CSVと対応しているか**の2点だけである。

| 確認すること | これが崩れると何が起きるか |
|---|---|
| 行数・列数が対象橋梁数と一致 | 橋の数と行列の大きさが合わず、割当が対応づかない |
| 対称かつ対角0 | 距離として成立しない |
| `order` が対象橋梁CSVのIDと一致 | 行列の行と橋の対応がずれ、別の橋の距離で制約がかかる |

いずれも対象地域を変えても成り立つべき条件で、特定の橋梁数や距離を前提にしていない。
最大距離は参考として表示するだけで判定には使わない（最適化の結果を前提にした判定を
上流の確認へ持ち込まないため）。

In [ ]:
# 保存済みの距離行列を読み、対象橋梁CSVと対応しているかを確かめる
dm = pickle.loads((REPO_ROOT / "data/processed/distance_matrix_322_20251208.pkl").read_bytes())
order, d_core = dm["order"], np.asarray(dm["d_core"])
print("pkl の中身:", list(dm.keys()), "| d_core:", d_core.shape, d_core.dtype)

# 大きさは対象橋梁CSVの件数から決まる（特定の数を書き込まない）
n_bridges = len(bridges)
check("3.距離行列", "行数・列数が対象橋梁数と一致",
      d_core.shape == (n_bridges, n_bridges), f"{d_core.shape} vs 橋梁 {n_bridges} 件")
check("3.距離行列", "対称かつ対角0",
      bool(np.allclose(d_core, d_core.T) and np.allclose(np.diag(d_core), 0)))
check("3.距離行列", "order が対象橋梁CSVのIDと一致",
      set(order) == set(bridges["shisetsu_id"]), f"order {len(order)} 件")

# 参考情報（判定には使わない）
dmax = float(d_core.max())
print(f"\n対象橋梁間の最大距離: {dmax:.2f} km")
print(f"  → 距離上限 D がこれを超えると、距離制約は実質的に働かなくなる。")
print(f"    本研究の結果で D≥40 km のとき全域が1つの管理エリアになるのはこのためである。")

### 4-2. 距離行列を作り直す　🟢`RUN_FROM_RAW = True` のときだけ

対象橋梁を作り直したときは、距離行列もその橋梁に合わせて作り直す必要がある。
`step3_build_distance_matrix.py` に新しい対象橋梁CSVを渡すと、
総当たりの大円距離を計算して新しい行列を出力する。

#### キャッシュの扱い

計算した距離はSQLiteに保存され、次回同じ座標の組が出てきたときに再利用される。
`--db-path` を指定しなければ**共有キャッシュに書き込まれる**ので、
対象橋梁を変えれば、その分の座標と距離が共有キャッシュに追加されていく。

このノートブックでは共有キャッシュを触らないよう、実行ごとの一時ファイル
（`OUT/dist_cache.sqlite`）を指定する。そのため毎回ゼロから計算するが、
322橋（総当たり51,681組）でも1秒程度で終わる。
共有キャッシュを育てたい場合は、ノートブックからではなくコマンドラインから
`--db-path` を省いて実行する。

#### 保存済みとの比較

できた行列は保存済みのものと比べる。並び順（`order`）は生成のたびに変わりうるため、
橋梁IDで対応づけてから距離の値を比較する。

- **一致した場合** — 同じ橋梁集合から同じ距離行列が再現できたことになる
- **一致しない場合** — 対象橋梁が変わったのであれば、それが期待どおりの動作である

対象を変えた場合は、この行列を保存済みのものと差し替え、最適化も解き直す必要がある。

In [ ]:
if RUN_FROM_RAW:
    # 新しい対象橋梁CSVから距離行列を作り直す（キャッシュは実行ごとの一時ファイル）
    run("step3_build_distance_matrix.py",
        "--input", RAW_OUT / "target_rc_bridges_322.csv",
        "--output", RAW_OUT / "distance_matrix.pkl",
        "--db-path", OUT / "dist_cache.sqlite")

    rebuilt = pickle.loads((RAW_OUT / "distance_matrix.pkl").read_bytes())
    r_order, r_core = list(rebuilt["order"]), np.asarray(rebuilt["d_core"])
    print(f"作り直した距離行列: {r_core.shape}, 最大距離 {r_core.max():.5f} km")

    # 距離として成立しているか（対象が変わっても成り立つべき条件）
    check("3.距離行列", "作り直した行列が正方・対称・対角0",
          bool(r_core.shape[0] == r_core.shape[1]
               and np.allclose(r_core, r_core.T) and np.allclose(np.diag(r_core), 0)))
    check("3.距離行列", "作り直した行列の並びが対象橋梁CSVと対応",
          set(r_order) == set(pd.read_csv(RAW_OUT / "target_rc_bridges_322.csv")["shisetsu_id"]))

    # 保存済みとの比較。並び順が違いうるのでIDで揃えてから比べる
    if set(r_order) == set(order):
        idx = [r_order.index(x) for x in order]
        diff = float(np.abs(r_core[np.ix_(idx, idx)] - d_core).max())
        print(f"\n→ 保存済みと同じ橋梁集合。距離の最大差 {diff:.2e} km")
        check("3.距離行列", "作り直した距離が保存済みと一致（並べ替え後）", diff < 1e-9,
              f"最大差 {diff:.2e} km")
    else:
        print("\n→ 保存済みとは異なる橋梁集合。対象を変えたのであれば期待どおり。")
        print("  これを以降の分析に使うには、保存済みの距離行列と差し替える:")
        print(f"    cp {(RAW_OUT / 'distance_matrix.pkl')}"
              f" data/processed/distance_matrix_322_20251208.pkl")
        print("  ※ 差し替えたら地域分割最適化も解き直す必要がある（第5章）。")
else:
    print("RUN_FROM_RAW = False のためスキップ（保存済みの距離行列を使う）")

## 5. ステージ4 — 地域分割最適化

対象橋梁を $M$ 個の管理エリアへ分け、期待契約件数の合計
$\sum_m f(N_m, L)$ を最小にする割当を求める。制約は「同じエリア内の
橋どうしの距離が上限 $D$ 以内」で、$D$ と $M$ を変えた全36通りを解いている。

この工程だけは商用ソルバー Gurobi を必要とし、計算時間も他とは桁が違う
（36通りで約2.3時間。うち $D=15$, $M=6$ の1件だけで2.1時間）。
そのため、解くか解かないかを `RUN_GUROBI` で選ぶ。

| | `RUN_GUROBI = True`（Gurobiがある環境） | `RUN_GUROBI = False`（既定） |
|---|---|---|
| すること | **実際に解く。その結果が答えである** | 解かない。保存済みの結果を使う |
| 保存済みとの比較 | 条件を変えていなければ同じ答えになるはず、という**再現性の確認** | — |
| 条件を変えたとき | 新しい答えが得られる。保存済みと違って当然 | 保存済みは古い条件のままなので使えない |

つまり、**解いたときはその出力が答え**であって、保存済みの結果はそれと突き合わせる
参照点にすぎない。対象橋梁や距離上限を変えた場合は、保存済みと一致しないのが正しい。

解かない場合は、保存済みの結果ファイルどうしの辻褄だけを確認する。
最適解かどうかはGurobiにしか判定できないので、ここで確認できるのは
**ファイルの取り違えや破損がないこと**だけである。

### 5-1. 最適化を解く　🟢Gurobi環境 ＋ `RUN_GUROBI = True` のときだけ

`run_gurobi_districting.py` に距離行列を渡し、指定した $(D, M)$ の組について解く。

解く組み合わせは `GUROBI_CASES` で指定する。既定は代表3ケース
（$D=25/M=3$、$D=35/M=3$、$D=40/M=1$）で、記録によればGurobiで合計3秒である。
ただしこの3ケースは**現在の対象地域と距離上限の設定における代表**にすぎない。
対象や $D$ の範囲を変えたときは、どのケースを見るべきかも合わせて見直す必要がある。
全36通りを解き直す場合は `GUROBI_CASES` にすべて並べる（約2.3時間）。

解いた結果は保存済みの結果と突き合わせる。

- **条件を変えていない場合** — 同じ答えが得られるはずで、一致すれば再現性が確認できる
- **条件を変えた場合** — 違う答えになるのが正しい。得られた結果を保存済みと差し替える

Gurobiが無い環境では自動でスキップする。

In [ ]:
RUN_GUROBI = False                          # True にすると解き直す（Gurobiが無ければ自動スキップ）
GUROBI_CASES = ["25:3", "35:3", "40:1"]     # 代表3ケース≒3秒。フルグリッド36ケースは約2.3時間

canon_csv = REPO_ROOT / "data/processed/optimization_results_exact_objective.csv"
canon = pd.read_csv(canon_csv).sort_values(["MaxDistance", "M"]).reset_index(drop=True)

if RUN_GUROBI and has_gurobi:
    run("run_gurobi_districting.py",
        "--pwl", "all", "--cases", *GUROBI_CASES,
        "--output", OUT / "gurobi_recheck.csv",
        "--solutions-output", OUT / "gurobi_recheck_solutions.pkl",
        "--no-archive", timeout=3600)
    redo = pd.read_csv(OUT / "gurobi_recheck.csv")
    merged = redo.merge(canon, on=["MaxDistance", "M"], suffixes=("_redo", "_canon"))
    merged["差"] = (merged["ObjectiveValue_Exact_redo"] - merged["ObjectiveValue_Exact_canon"]).abs()
    display(merged[["MaxDistance", "M", "ObjectiveValue_Exact_redo",
                    "ObjectiveValue_Exact_canon", "RegionCounts_redo", "差"]])
    check("4.最適化", "Gurobiで解き直した代表ケースが正本と厳密一致",
          bool((merged["差"] < 1e-9).all()) and len(merged) == len(GUROBI_CASES),
          f"{len(merged)} ケース, 最大差 {merged['差'].max():.2e}")
elif RUN_GUROBI:
    print("gurobipy が無いためスキップ。Gurobiのある環境（Windows側）で実行してください。")
else:
    print("RUN_GUROBI = False のためスキップ（正本の最適化結果をそのまま使う）")

### 5-2. 保存済み結果の辻褄を確認する

解かない場合（および解いた結果を保存済みと比べる前提として）、
保存済みの結果ファイルが内部で矛盾していないかを見る。

結果CSVには「地域ごとの橋梁数（`RegionCounts`）」と「目的関数値
（`ObjectiveValue_Exact`）」の両方が記録されている。前者から $f(N_m, L)$ を
組み直して後者になるかを確かめる。

同じ割当・同じ $q$・同じ $L$ なら一致して当然の照合であり、
**最適かどうかを確かめているわけではない**。狙いは、結果CSVと $q$ と $L$ が
同じ実行の組み合わせであることの確認である。第3章で確認したのは $q$ 自体の出所で、
ここで見るのはその $q$ が最適化結果と組になっているかという別の点にあたる。

In [ ]:
check("4.最適化", "36ケース（D 8水準 × M）が揃っている", len(canon) == 36, f"{len(canon)} 行")
check("4.最適化", "全ケースが L=5 で解かれている", set(canon["BundleLimit"]) == {5}, str(set(canon["BundleLimit"])))
check("4.最適化", "全ケースで地域規模の合計が322",
      all(sum(int(x) for x in s.split(";")) == 322 for s in canon["RegionCounts"]))

# 再評価ロジックの検算: RegionCounts から f を組み直すと ObjectiveValue_Exact になるはず
recomputed = [sum(expected_contracts(int(n), 5, q) for n in map(int, s.split(";")))
              for s in canon["RegionCounts"]]
err = float(np.abs(np.array(recomputed) - canon["ObjectiveValue_Exact"].to_numpy()).max())
check("4.最適化", "正本CSVの目的関数値が地域規模から閉形式で再現できる", err < 1e-9, f"最大差 {err:.2e}")
display(canon[["MaxDistance", "M", "ObjectiveValue_Exact", "RegionCounts"]].head(8))

### 5-3. 割当pklとCSVが同じ実行のものか

`districting_solutions_all36.pkl` には、Gurobiが出した**橋梁ごとの地域割当**（322×M の0/1行列）が入っている。
ここから地域規模を数え直し、$f$ を組み直して正本CSVと一致するかを全36ケースで確認する。

これも一致して当たり前の照合だが、地図（代表図・atlas）が使っているのはCSVではなく**この割当**なので、
**本文の図と表が同じ解を見ている**ことの担保になる。最適化を解き直したのに
割当pklだけ差し替え忘れる、といった取り違えをここで検出できる。

In [ ]:
# 割当pkl から目的関数値を独立に再計算して結果CSVと突合（atlas/代表図の根拠と同じ検証）
# 読み込みは地図描画と同じ load_solutions を使う（{(D, M): 322×M の0/1割当行列} に正規化される）
from bundling_analysis.districting_map import load_solutions

solutions = load_solutions(REPO_ROOT / "data/processed/districting_solutions_all36.pkl")
print("割当pkl のケース数:", len(solutions), "| 例:", sorted(solutions)[:3])
print("割当行列の形:", solutions[(25.0, 3)].shape, "（行=橋梁, 列=地域）")

rows = []
for (d, m), assign in sorted(solutions.items()):
    sizes = sorted((int(assign[:, r].sum()) for r in range(assign.shape[1])), reverse=True)
    f_val = sum(expected_contracts(int(n), 5, q) for n in sizes)
    ref = canon[(canon.MaxDistance == d) & (canon.M == m)]
    ref_sizes = sorted((int(x) for x in ref["RegionCounts"].iloc[0].split(";")), reverse=True)
    rows.append({"D": d, "M": m, "sizes_pkl": sizes, "sizes_csv": ref_sizes,
                 "f_pkl": f_val, "f_csv": float(ref["ObjectiveValue_Exact"].iloc[0]),
                 "一致": sizes == ref_sizes and abs(f_val - float(ref["ObjectiveValue_Exact"].iloc[0])) < 1e-9})
verify = pd.DataFrame(rows)
display(verify.head(6))
check("4.最適化", "割当pklとCSVが全36ケースで同じ解を指している",
      bool(verify["一致"].all()), f"{int(verify['一致'].sum())}/{len(verify)} ケース")

## 6. ステージ5 — 図表の生成

ここから下は、これまでに用意した3つの入力（対象橋梁CSV・$q$・最適化結果）から
論文の図と表を作る。すべてリポジトリ内のデータだけで動く。

出力先は `FIG_DIR`・`TAB_DIR` で、既定では作業用フォルダ、
`UPDATE_CANONICAL = True` のときは論文が参照する `figures/`・`outputs/` になる。

### 論文の図との対応

本文の図は次の6点で、いずれもこの章のどこかのセルで生成・表示される。
本文での並び順とこの章の順序は一致しない（作る順序は上流からの依存関係で決まるため）。

| 本文 | 内容 | ファイル | 作る節 |
|---|---|---|---|
| Figure 1 | 対象6市町と322橋の分布 | `study_area.png` | 6-4 |
| Figure 2 | 点検間隔の分布 | `inspection_interval.png` | 3-4（第3章で作成） |
| Figure 3 | $f(N,L)$ の形状 | `expected_contracts_by_bundle_limit.png` | 6-3 |
| Figure 4 | 距離上限と地域数に対する期待契約件数 | `dm_sensitivity.png` | 6-2 |
| Figure 5 | 代表解の地域別内訳 | `region_breakdown.png` | 6-2 |
| Figure 6 | 代表3ケースの地域分割図 | `districting_map_D25_M3/D35_M3/D40_M1.png` | 6-4 |

このほか、本文には未挿入だが生成しているものがある。

| ファイル | 位置づけ |
|---|---|
| `expected_contracts_scaling_analysis.png` | 第4.3節向けに作成。本文へは未挿入 |
| `districting_atlas.png` | 全36ケースの一覧。付録・電子補足資料向け |
| `optimization_results.png` | 本文では Figure 4 を採用したため未使用。ただし同じプログラムが表4.2のLaTeX行を出力するので実行は必要 |

### 6-1. 距離上限ごとの最適値と、現行管理との比較表

`plot_optimization_results.py` が2つの成果物を作る。

- **図** — 距離上限 $D$ ごとに、期待契約件数が最小になる地域数 $M$ を選び、その最小値を並べた折れ線
- **表** — 現行管理（市町ごとに発注する場合）と、最適化した場合の期待契約件数・低減率

比較の基準になる「現行管理」の値は、あらかじめ書かれた数値ではなく、
対象橋梁CSVの管理者別橋梁数から $\sum_m f(N_m, L)$ をその場で計算したものである。
市町の内訳が変われば基準値も自動的に変わる。

このセルでは次の2点を確かめる。

| 確認すること | 意味 |
|---|---|
| 基準値が既知の値と一致 | 対象橋梁CSVと $q$ と $L$ の組み合わせが論文執筆時と同じ |
| 基準値が管理者別内訳から再現できる | 表の基準値が別経路の値ではなく、この場の計算で出ている |

本文に貼り付けられる形式のLaTeX行も出力されるので、条件を変えて計算し直したときは
その出力をそのまま差し替えに使える。

In [ ]:
# fig:optimization_results / tab:optimization_results
out = run("plot_optimization_results.py",
          "--output-stem", FIG_DIR / "optimization_results",
          "--table-output", TAB_DIR / "optimization_results_table.csv")

tab = pd.read_csv(TAB_DIR / "optimization_results_table.csv")
baseline = float(tab["ObjectiveValue_Exact"].iloc[0])   # 先頭行＝現行管理（管理者別）
best = float(tab["ObjectiveValue_Exact"].min())
print(f"現行管理の基準値 = {baseline:.6f} / 最良解 = {best:.6f} / 低減率 = {(1-best/baseline)*100:.1f}%")

check("5.図表", "現行管理の基準値が統一系列の 2.588321", abs(baseline - 2.588321070916122) < 1e-9,
      f"{baseline:.6f}")
check("5.図表", "基準値が管理者別橋梁数から閉形式で再現できる",
      abs(baseline - sum(expected_contracts(int(n), 5, q) for n in counts)) < 1e-12)
show(FIG_DIR / "optimization_results.png", width=620)

### 6-2. 感度と内訳を見る2枚

6-1の図は距離上限ごとの「最良の $M$」だけを示すため、地域数を変えたときの挙動が見えない。
そこを補うのが次の2枚である。

- **`plot_dm_sensitivity.py`** — 地域数 $M$ ごとに1本ずつ線を引き、距離上限 $D$ に対する
  期待契約件数の変化を示す。解けなかった $(D, M)$ の組は線をつながず空けたままにする
  （値を補間したり0で埋めたりしない）
- **`plot_region_breakdown.py`** — 代表的な解について、期待契約件数を地域ごとの寄与に分解する。
  大きな地域に橋梁を集められるかどうかが効いていることが読み取れる

どちらも入力は保存済みの最適化結果CSVで、$q$ は第3章で確認した推移確率行列から導かれる。

In [ ]:
# fig:dm_sensitivity（D×M感度）と fig:region_breakdown（代表解の地域内訳）
run("plot_dm_sensitivity.py", "--output-stem", FIG_DIR / "dm_sensitivity")
run("plot_region_breakdown.py",
    "--output-stem", FIG_DIR / "region_breakdown",
    "--table-output", TAB_DIR / "region_breakdown.csv")

display(pd.read_csv(TAB_DIR / "region_breakdown.csv"))
show(FIG_DIR / "dm_sensitivity.png", width=620)
show(FIG_DIR / "region_breakdown.png", width=620)

### 6-3. 期待契約件数そのものの性質を見る2枚

ここまでの図は最適化の結果を示すものだったが、この2枚は最適化を通さず、
期待契約件数の関数 $f(N, L)$ が持つ性質を直接描く。

- **`plot_expected_contracts_by_limit.py`** — 地域内の橋梁数 $N$ に対して $f(N,L)$ がどう増えるか。
  $L=1$（1件ずつ発注）では $Nq$ に比例するが、$L$ が大きいほど増え方が緩やかになる
- **`plot_expected_contracts_scaling_analysis.py`** — 2枚組。
  (a) 同じ年に2件以上の補修需要が同じ地域で発生する確率（これが起きないと束ねる相手がいない）、
  (b) 1件ずつ発注した場合に対する削減率と、その理論上限 $1 - 1/L$

後者は $N$・$L$・$q$ をすべて既存の入力から導出する。橋梁数は対象橋梁CSVの件数から、
$L$ は最適化結果CSVに記録された値から取るので、図の中に手入力の数値はない。

そこで次の2点を確かめる。

| 確認すること | 意味 |
|---|---|
| 図と一緒に出力されたCSVの $q$ が第3章の推定値と一致 | 図が古い $q$ で描かれていない |
| $f(322,5)$ が最適化結果の「全域を1地域にした場合」と一致 | 同じ関数が最適化でも作図でも使われている |

In [ ]:
# 第4.3節の2枚（どちらも expected_contracts() だけを計算根拠にしている）
run("plot_expected_contracts_by_limit.py", "--output-stem", FIG_DIR / "expected_contracts_by_bundle_limit")
run("plot_expected_contracts_scaling_analysis.py",
    "--output-stem", FIG_DIR / "expected_contracts_scaling_analysis",
    "--csv-output", TAB_DIR / "expected_contracts_scaling_analysis.csv")

# float_precision="round_trip" が要る: pandas既定のfloatパーサはCSVの最終桁を落とし、
# q が 0.0123297879741142 になって元の値と 5.7e-17 ずれる（値の追跡が効かなくなる）
scal = pd.read_csv(TAB_DIR / "expected_contracts_scaling_analysis.csv", float_precision="round_trip")
row322 = scal[scal.N == 322].iloc[0]
print(f"q(CSV) = {row322['repair_probability_q']!r} / 行列 = {row322['transition_matrix_name']}")
print(f"N=322: f(322,5) = {row322['expected_contracts_L5']:.9f}, 削減率 = {row322['reduction_rate_L5']:.6f}")

check("5.図表", "感度分析CSVの q がステージ2で再推定した q と一致",
      abs(float(row322["repair_probability_q"]) - q) < 1e-15)
check("5.図表", "感度分析の f(322,5) が最適化結果の D≥40,M=1（全域1地域）と一致",
      abs(float(row322["expected_contracts_L5"])
          - float(canon[(canon.MaxDistance >= 40) & (canon.M == 1)]["ObjectiveValue_Exact"].iloc[0])) < 1e-9)
show(FIG_DIR / "expected_contracts_by_bundle_limit.png", width=620)
show(FIG_DIR / "expected_contracts_scaling_analysis.png", width=900)

### 6-4. 地図（対象地域図・代表地域分割図・atlas）

対象地域図と、代表3ケースの地域分割図、全36ケースを並べたatlasを描く。
行政界データを読むため `geopandas` が必要で、約20秒かかる。
不要なら `RUN_MAPS = False` にして飛ばせる。

`make_districting_maps.py` は描画のたびに、割当と結果CSVの地域規模・目的関数値を
照合する。その記録（`atlas.meta.json`）を読んで、全ケースが照合済みかを確認する。

行政界ファイルの読み込みは環境によって経路が変わる。古い `geopandas`（〜0.11）は
`fiona` 1.10 で廃止された関数を参照して失敗するため、その場合は `pyogrio` で
読み直すようにしてある（`src/bundling_analysis/admin_boundary.py`）。
それでも失敗する環境では、ノートブック全体は止めず、この節だけを失敗として
記録して先へ進む。

In [ ]:
# 地図系（geopandas 必須・約20秒）。不要なら RUN_MAPS = False
RUN_MAPS = True

if RUN_MAPS:
    try:
        run("plot_study_area_map.py", "--output-stem", FIG_DIR / "study_area")
        run("make_districting_maps.py",
            "--solutions", REPO_ROOT / "data/processed/districting_solutions_all36.pkl",
            "--fig-dir", FIG_DIR, "--atlas-dir", FIG_DIR / "atlas")
        meta = json.loads((FIG_DIR / "atlas/atlas.meta.json").read_text())
        ver = pd.DataFrame(meta["verifications"])
        print(f"atlas.meta.json の検証レコード: {len(ver)} 件 / verified=True {int(ver['verified'].sum())} 件")
        show(FIG_DIR / "study_area.png", width=560)
        # 代表3ケースはいずれも本文で使う図なので3枚とも表示する
        for d, m in [(25, 3), (35, 3), (40, 1)]:
            print(f"\n代表ケース D={d} km, M={m}")
            show(FIG_DIR / f"districting_map_D{d}_M{m}.png", width=560)
        show(FIG_DIR / "atlas/districting_atlas.png", width=900)
        check("5.図表", "地図の全ケースが結果CSVと照合済み（地域規模・目的関数値）",
              bool(ver["verified"].all()) and len(ver) >= 36, f"{int(ver['verified'].sum())}/{len(ver)} 件")
    except RuntimeError as e:
        # geopandas 系の環境不整合でここだけ失敗しても、以降のセルは実行できる
        check("5.図表", "地図の生成", False, f"{e}（上のエラー出力を参照）")
        print("\n地図の生成に失敗した。多くは geopandas と依存ライブラリの版の組み合わせが原因:")
        print("  現在の環境:", sys.executable)
        print("  確認: python -c \"import geopandas, fiona; print(geopandas.__version__, fiona.__version__)\"")
        print("  対処例: conda install -c conda-forge 'geopandas>=1.0'   （または fiona<1.10 に固定）")
        print("  地図が不要なら RUN_MAPS = False にして先へ進める。")
else:
    print("RUN_MAPS = False のためスキップ")

## 7. 章をまたいだ突き合わせ

ここまでは各章の中で確認をしてきたが、章をまたぐと同じ値が別の経路で計算される場面がある。
同じ意味の値が本当に同じになっているかを、最後にまとめて確かめる。

確認するのは次の4点。

1. **$q$ が全経路で一致するか** — 推定用データから再推定した値、`expected_contracts.py` が
   読み込んだ値、感度分析の出力CSVに記録された値の3つを比べる。
   CSVを経由すると桁が落ちることがあるため、読み込み時に丸めが起きない設定を使う
2. **対象橋梁数が全経路で一致するか** — 橋梁CSVの行数、距離行列の大きさ、
   最適化結果の地域規模の合計、割当行列の行数の4つを比べる
3. **期待契約件数の実装が外部の表と一致するか** — 以前に受領した $f(N,L)$ の一覧表と、
   `expected_contracts()` の計算結果を突き合わせる。この表は現在より前の $q$ で作られているため、
   完全一致ではなく「その差で説明できる範囲か」を見る
4. **論文で使う主要な数値** — $q$、現行管理の基準値、最良解、低減率などを一覧表示する

4の表にある「低減率」と「削減率」は分母が異なる別の指標なので、
本文で使うときはどちらの基準かを明示する必要がある。

In [ ]:
# (1) q は全経路で同一か
from bundling_analysis.expected_contracts import repair_probability_from_transition_matrix as rp

q_sources = {
    "eMarkov再推定 (markov_input から)": float(q_rerun),
    "DEFAULT_TRANSITION_MATRIX": float(rp(DEFAULT_TRANSITION_MATRIX)[1]),
    "感度分析CSV (図4.3の実行時に記録)": float(row322["repair_probability_q"]),
}
display(pd.DataFrame({"q（全桁）": {k: repr(v) for k, v in q_sources.items()}}))
spread = max(q_sources.values()) - min(q_sources.values())
check("8.横断", "q が全経路でビット一致", spread == 0.0, f"最大差 {spread:.1e}")

# (2) N=322 は全経路で同一か
n_sources = {"橋梁CSV行数": len(bridges), "距離行列サイズ": int(d_core.shape[0]),
             "結果CSVのRegionCounts合計": sum(int(x) for x in canon["RegionCounts"].iloc[0].split(";")),
             "割当行列の行数": int(solutions[(25.0, 3)].shape[0])}
display(pd.Series(n_sources, name="N").to_frame())
check("8.横断", "N=322 が全経路で一致", len(set(n_sources.values())) == 1 and len(bridges) == 322)

# (3) 独立ソース（受領済みの期待契約件数テーブル）と自前の閉形式が一致するか。
#     この表は採用qより前の q で計算されているので、完全一致ではなく「q差で説明できる範囲か」を見る。
mat = pd.read_csv(REPO_ROOT / "data/processed/expected_contracts_matrix_n1-322.csv")
err = max(abs(expected_contracts(int(n), L, q) - float(mat.loc[mat.N == n, f"L={L}"].iloc[0]))
          for L in (1, 3, 5, 7, 10) for n in mat["N"])
q_table = float(mat.loc[mat.N == 322, "L=1"].iloc[0]) / 322      # L=1 は N*q なので逆算できる
print(f"表から逆算した q = {q_table!r}\n採用している q   = {q!r}\n差 = {abs(q_table - q):.2e}")
check("8.横断", "独立ソースの表と閉形式が一致（表側の丸め＋旧qの差の範囲）",
      err < 1e-4, f"最大差 {err:.2e}（N=322, L=1 で最大）")
check("8.横断", "表の裏にある q と採用 q の差が微小（系列の違いは1e-7オーダー）",
      abs(q_table - q) < 1e-6, f"{abs(q_table - q):.2e}")

# (4) 論文本文で使う主要数値
main = {
    "q": q,
    "現行管理の基準値": baseline,
    "最良解 f (D≥40, M=1)": float(canon["ObjectiveValue_Exact"].min()),
    "低減率 vs 現行管理": 1 - float(canon["ObjectiveValue_Exact"].min()) / baseline,
    "個別発注 N*q": 322 * q,
    "削減率 vs 個別発注 (L=5)": float(row322["reduction_rate_L5"]),
}
display(pd.Series(main, name="値").to_frame())

> **本文執筆時の注意**: 上の表の「低減率 vs 現行管理（0.539）」と「削減率 vs 個別発注（0.699）」は
> どちらも同じ $f(322,5)=1.1933$ から出るが**分母が違う**（2.5883 と 3.9702）。
> 4.3節と4.5節で別々の基準を使っているので、本文では毎回どちらの基準か明示する。

## 8. main.tex への反映

`main.tex` には `\graphicspath` が無いため、`\includegraphics{figures/...}` は
**texファイルからの相対パス**、つまり `paper/latex/figures/` を読む。
作図スクリプトの出力先はリポジトリ直下の `figures/` なので、両者は別のディレクトリで、
**回し直した図をtexへ届けるにはコピーが要る**。

### 8-1. texが参照している図の状態を調べる

`main.tex` を読んで `\includegraphics` のパスを全部抜き出し、次の3点を突き合わせる。
このセルは読むだけで、何も書き換えない。

- tex側（`paper/latex/figures/`）にファイルがあるか
- 生成先（`figures/`）に対応するファイルがあるか
- 両者が同一か（md5）。違えば **texが古い図で組版されている**

本文はベクタ形式（PDF）で図を参照する。ラスタ画像（PNG）だと拡大時に粗くなり、
学術誌の投稿規定にも合わないため。作図スクリプトは同じ図を PNG・PDF・SVG で出力しており、
`main.tex` はそのうち PDF を読む。PDF 内の文字は TrueType で埋め込まれる
（`plotting_utils.setup_figure_defaults()`）。既定の Type 3 フォントは多くの学術誌が
受け付けないため、`tests/test_figure_fonts.py` で混入を検査している。

In [ ]:
import hashlib
import re

TEX = REPO_ROOT / "paper/latex/main.tex"
TEX_DIR = TEX.parent
GEN_DIR = REPO_ROOT / "figures"


def md5(path: Path) -> str:
    return hashlib.md5(path.read_bytes()).hexdigest()


refs = re.findall(r"\\includegraphics(?:\[[^\]]*\])?\{([^}]*)\}", TEX.read_text(encoding="utf-8"))
rows = []
for ref in refs:
    tex_side = TEX_DIR / ref              # texが実際に読むファイル
    gen_side = GEN_DIR / Path(ref).name   # 作図スクリプトが書くファイル
    state = ("tex側に無い" if not tex_side.exists()
             else "生成先に無い" if not gen_side.exists()
             else "最新" if md5(tex_side) == md5(gen_side) else "★古い")
    rows.append({"main.texの参照": ref, "状態": state})

tex_figs = pd.DataFrame(rows)
display(tex_figs)

stale = tex_figs[tex_figs["状態"] != "最新"]
check("8.tex反映", "main.texが参照する図がすべて最新の生成物と一致",
      stale.empty, f"{len(tex_figs) - len(stale)}/{len(tex_figs)} 枚が最新"
      + ("" if stale.empty else f" / 要同期: {list(stale['main.texの参照'])}"))

# 生成したがtexに未挿入の図（形式違いは同一視するため拡張子を除いて比べる）
used = {Path(r).stem for r in refs}
unused = sorted({p.stem for p in GEN_DIR.glob("*.png")} - used)
print("\ntexに未挿入の図:", unused or "なし")

### 8-2. 図をtex側へコピーする

上で「★古い」「tex側に無い」と出たものを `figures/` から `paper/latex/figures/` へ複製する。
**main.tex は一切書き換えない。**

既定は `SYNC_TEX_FIGURES = False`（何もしない）。コピーしたいときだけ `True` にする。
コピー対象は main.tex が実際に参照しているファイルだけなので、
未挿入の図が紛れ込むことはない。

In [ ]:
import shutil

SYNC_TEX_FIGURES = False   # True にすると paper/latex/figures/ へコピーする

if SYNC_TEX_FIGURES:
    copied = []
    for ref in refs:
        tex_side = TEX_DIR / ref
        gen_side = GEN_DIR / Path(ref).name
        if not gen_side.exists():
            print(f"生成先に無いのでスキップ: {gen_side.relative_to(REPO_ROOT)}")
            continue
        if tex_side.exists() and md5(tex_side) == md5(gen_side):
            continue
        tex_side.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(gen_side, tex_side)
        copied.append(ref)
    print(f"コピーした図: {len(copied)} 枚")
    for ref in copied:
        print(f"  figures/{Path(ref).name} → paper/latex/{ref}")

    after = [ref for ref in refs
             if (TEX_DIR / ref).exists() and (GEN_DIR / Path(ref).name).exists()
             and md5(TEX_DIR / ref) == md5(GEN_DIR / Path(ref).name)]
    check("8.tex反映", "同期後、main.texが参照する図がすべて最新",
          len(after) == len(refs), f"{len(after)}/{len(refs)} 枚")
else:
    print("SYNC_TEX_FIGURES = False のためコピーしない（11-1の表で状態だけ確認）")

### 8-3. この実行の記録を残す

`make_all_figures.py` が `run.meta.json` に残していたのと同じ情報
（実行時刻・git HEAD・各スイッチ・生成物一覧・チェック結果）を、
このノートの実行フォルダにも書き出す。作図をノート側で回しても実行ログが残るようにするため。

In [ ]:
meta = {
    "generated_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "git_head": subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT,
                               capture_output=True, text=True).stdout.strip(),
    "notebook": "notebooks/pipeline_walkthrough.ipynb",
    "switches": {
        "RUN_FROM_RAW": RUN_FROM_RAW,
        "RUN_GUROBI": RUN_GUROBI,
        "RUN_MAPS": RUN_MAPS,
        "UPDATE_CANONICAL": UPDATE_CANONICAL,
        "SYNC_TEX_FIGURES": SYNC_TEX_FIGURES,
    },
    "output_dirs": {"figures": str(FIG_DIR.relative_to(REPO_ROOT)),
                    "tables": str(TAB_DIR.relative_to(REPO_ROOT))},
    "checks": {"total": len(CHECKS),
               "failed": [c for c in CHECKS if c["result"] == "FAIL"]},
    "artifacts": sorted(str(p.relative_to(REPO_ROOT)) for p in FIG_DIR.glob("*.png")),
}
(OUT / "run.meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(meta["switches"], ensure_ascii=False, indent=2))
print(f"\n記録: {(OUT / 'run.meta.json').relative_to(REPO_ROOT)}")

## 9. まとめ

### このノートブックで確認していること

点検データから論文の図表までの各段階について、入力・出力・つながりを実際に実行して確かめる。
確認結果は最後の一覧表にまとまる。主なものは次のとおり。

| 段階 | 確認していること |
|---|---|
| 対象橋梁 | 下流の計算が前提とする条件を満たすか。原データから作り直した場合は何が変わったか |
| 劣化推定 | 推定用データから再推定した $q$ が、下流が実際に使う値と一致するか |
| 距離行列 | 距離行列として成立し、対象橋梁と対応づいているか |
| 最適化 | （解く場合）解いた結果と保存済みが一致するか。（解かない場合）保存済み同士の辻褄が合うか |
| 図表 | 図表に使われた $q$・$N$・$L$ が上流と同じか |
| 本文 | `main.tex` が読む図が最新の生成物と一致しているか |

### 現在の状態

コード側で対応した項目:

- $q$ の根拠をリポジトリ内に置き、`expected_contracts.py` がそれを読むようにした。
  推定と利用が繋がり、ずれれば警告とテストで検出される
- `reevaluate_optimization_objectives.py` の既定値が正本を壊す問題を解消した
  （`--input`・`--output` を必須化し、列が失われる上書きは `--force` なしで拒否）
- 旧系列の結果CSVを管理対象から外し、それに伴う照合工程も削除した
- 遷移集計表と旧docxの差分表示を廃止した（`main.tex` は現行値に更新済みのため）

本文・運用側に残る項目:

1. 4.3節の「削減率（1件ずつ発注した場合との比）」と 4.5節の「低減率（現行管理との比）」は
   分母が異なる。本文では毎回どちらの基準かを明示する
2. `main.tex` が読む図は `paper/latex/figures/` に置く必要があり、
   図を作り直したら第8章で同期する（本文はPDFを参照する）

### 参考

- 現行パイプラインから外れているコードの一覧: `docs/legacy_and_unused.md`
- 各プログラムの引数: `docs/cli_scripts_guide.md`
- データの由来: `data/README.md`

In [ ]:
summary = pd.DataFrame(CHECKS)
n_fail = int((summary.result == "FAIL").sum())
display(summary)
print(f"\n{len(summary) - n_fail} / {len(summary)} 件 PASS" + ("" if n_fail == 0 else f" / ❌ FAIL {n_fail} 件"))
print(f"このノートの出力: {OUT.relative_to(REPO_ROOT)}（git管理外・正本は無変更）")

summary.to_csv(OUT / "walkthrough_checks.csv", index=False)
print(f"チェック結果を保存: {(OUT / 'walkthrough_checks.csv').relative_to(REPO_ROOT)}")

### 9-2. スクリプト入出力インベントリ

各スクリプトの「入力 → 出力」と再実行可否の一覧。
どのファイルを触ると何が壊れるかを見るときの参照表。

In [ ]:
# スクリプト入出力の一覧（存在確認つき）
inventory = pd.DataFrame([
    ("1", "step3_extract_rc_bridges.py", "🟡", "x-Road原データCSV + 行政界shp", "宮城県RC橋CSV"),
    ("1", "step3_filter_target_municipalities.py", "🟡", "宮城県RC橋CSV", "data/processed/target_rc_bridges_322.csv"),
    ("2", "step3_prepare_markov_input.py", "🟡", "RC橋CSV + 道路メンテナンス年報", "markov_input_*.txt（4シナリオ）"),
    ("2", "step3_run_emarkov.py", "🟢", "markov_input_*.txt", "推移確率行列CSV + repair_probability.json"),
    ("2", "make_transition_counts.py", "🟢", "markov_input_with_supply.txt", "outputs/transition_counts.csv（tab:transition_counts）"),
    ("2", "plot_inspection_interval.py", "🟢", "markov_input_with_supply_collapse.txt", "figures/inspection_interval.*"),
    ("3", "step3_build_distance_matrix.py", "🟡", "322橋CSV + dist_cache.sqlite", "distance_matrix_322_20251208.pkl"),
    ("4", "run_gurobi_districting.py", "🟡", "距離行列pkl（Gurobi必須）", "最適化結果CSV + 割当pkl"),
    ("4", "reevaluate_optimization_objectives.py", "🟢", "Gurobi結果CSV（--input必須）", "optimization_results_exact_objective.csv"),
    ("5", "plot_optimization_results.py", "🟢", "結果CSV + 322橋CSV", "figures/optimization_results.* + LaTeX表"),
    ("5", "plot_dm_sensitivity.py", "🟢", "結果CSV + 322橋CSV", "figures/dm_sensitivity.*"),
    ("5", "plot_region_breakdown.py", "🟢", "結果CSV", "figures/region_breakdown.* + outputs/region_breakdown.csv"),
    ("5", "plot_expected_contracts_by_limit.py", "🟢", "DEFAULT_TRANSITION_MATRIX のみ", "figures/expected_contracts_by_bundle_limit.*"),
    ("5", "plot_expected_contracts_scaling_analysis.py", "🟢", "行列 + 322橋CSV + 結果CSV", "figures/expected_contracts_scaling_analysis.* + CSV"),
    ("5", "plot_study_area_map.py", "🟢", "322橋CSV + 市町境界geojson", "figures/study_area.*"),
    ("5", "make_districting_maps.py", "🟢", "割当pkl + 境界 + 距離行列 + 結果CSV", "代表地図3枚 + atlas"),
], columns=["ステージ", "スクリプト", "種別", "入力", "出力"])

pd.set_option("display.max_colwidth", 70)
display(inventory)